# Chat with a small language model in your browser

This notebook runs a real language model **entirely on your own computer**, inside the browser. Nothing is sent to a server.

**Before you start:** use Chrome or Edge on a laptop, and be on good wifi. Models download once and are then saved (cached) in your browser, so you never have to download the same model twice.

**The plan:** see what models exist, download one (or more), check what you have saved, then load one and chat with it.

## Step 1: Get set up

Three small cells: bring in the Python tools we need, check that your browser can run models, and load the WebLLM library.

### 1a: Import tools

- `pandas` (as `pd`) makes and filters tables.
- `json` turns Python data into text and back. We use it later to hand data to JavaScript.
- `pyjs` connects Python to the browser's **JavaScript**, the language web pages run on. WebLLM is written in JavaScript, so we need this bridge.
- `pprint` ("pretty print") prints big nested data neatly, one item per line.
- `display` lets a cell show something and then update it in place (we use it for a progress bar).

In [ ]:
import json
import pandas as pd
import pyjs
from pprint import pprint
from IPython.display import display

js = pyjs.js  # a doorway from Python into the browser's JavaScript

print("Tools imported.")

### 1b: Check for WebGPU

Models run on your computer's graphics chip (GPU). Browsers reach the GPU through a feature called **WebGPU**. Recent Chrome and Edge on a laptop have it; many phones and older browsers do not.

In [ ]:
if getattr(js.navigator, "gpu", None) is not None:
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

### 1c: Load WebLLM

**WebLLM** is a free JavaScript library that runs language models inside the browser. This line fetches it from the internet (it takes a few seconds the first time).

In [ ]:
webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

## Step 2: See which models are available

WebLLM comes with a long list of ready-to-use models. We will turn that list into a table, add a few useful columns, and then filter it down to the models worth trying.

### 2a: Look at WebLLM's model list

WebLLM keeps its list in `webllm.prebuiltAppConfig.model_list`. Each entry describes one model. Let's count them and look at the first one.

In [ ]:
model_list = list(webllm.prebuiltAppConfig.model_list)
print(f"WebLLM knows about {len(model_list)} models.\n")

first = model_list[0]
print("model_id:        ", first.model_id)
print("vram_required_MB:", first.vram_required_MB)

### 2b: Turn the list into a table

We keep three things about each model:

- **`name`**: the model's ID, which we use to download and load it.
- **`size_mb`**: how much graphics memory it needs, in megabytes (MB). 1000 MB is about 1 GB.
- **`type`**: what kind of model it is. WebLLM stores this as a number, so `TYPES` turns the number into a word:
  - `"chat"` models answer questions. This is what we use in this notebook.
  - `"embedding"` models turn text into lists of numbers, used for search. We will use these later.
  - `"vision"` models can look at images.

In [ ]:
TYPES = {0: "chat", 1: "embedding", 2: "vision"}

rows = []
for m in model_list:
    size = getattr(m, "vram_required_MB", None)
    rows.append({
        "name": m.model_id,
        "size_mb": round(size) if size else None,
        "type": TYPES.get(getattr(m, "model_type", None) or 0, "other"),
    })

table = pd.DataFrame(rows)
table["size_mb"] = table["size_mb"].astype("Int64")  # whole numbers, blank if unknown
table.head()

### 2c: Add release dates

WebLLM does not record when each model came out, so we keep our own small table, `RELEASED`, with the month each model **family** was first released (`"YYYY-MM"`). A model gets the date of the first family its name starts with.

These dates are approximate and typed in by hand, so this table **will fall behind** as new models are added to WebLLM. New models show up with a `?`. To fix one, add a line like `("NewFamily-", "2026-05"),` near the top. Put more specific names before less specific ones (`Qwen3.5-` before `Qwen3-`), because the first match wins.

In [ ]:
RELEASED = [
    ("Qwen3.5-", "2026-03"),
    ("Ministral-3-", "2025-12"),
    ("Qwen3-", "2025-04"),
    ("OLMo-2-0425", "2025-04"),
    ("gemma3-", "2025-03"),
    ("Phi-4-mini", "2025-02"),
    ("DeepSeek-R1-Distill", "2025-01"),
    ("SmolLM2-", "2024-11"),
    ("OLMo-2-1124", "2024-11"),
    ("Qwen2.5-Coder", "2024-11"),
    ("Qwen2.5-", "2024-09"),
    ("Llama-3.2-", "2024-09"),
    ("Phi-3.5", "2024-08"),
    ("Hermes-3", "2024-08"),
    ("gemma-2-", "2024-07"),
    ("Llama-3.1-", "2024-07"),
    ("Qwen2-", "2024-06"),
    ("Mistral-7B-Instruct-v0.3", "2024-05"),
    ("Llama-3-", "2024-04"),
    ("Phi-3-", "2024-04"),
    ("snowflake-arctic-embed", "2024-04"),
    ("Hermes-2-Theta", "2024-05"),
    ("Hermes-2-Pro", "2024-03"),
    ("gemma-2b-it", "2024-02"),
    ("TinyLlama", "2024-01"),
    ("stablelm-2-zephyr", "2024-01"),
    ("phi-2", "2023-12"),
    ("Mistral-7B-Instruct-v0.2", "2023-12"),
    ("NeuralHermes-2.5", "2023-12"),
    ("WizardMath-7B-V1.1", "2023-12"),
    ("OpenHermes-2.5", "2023-11"),
    ("phi-1_5", "2023-09"),
    ("Llama-2", "2023-07"),
    ("RedPajama", "2023-05"),
]


def released(name):
    for prefix, month in RELEASED:
        if name.lower().startswith(prefix.lower()):
            return month
    return "?"


table["released"] = table["name"].apply(released)
table.head()

### 2d: Mark which models are already saved

Once you download a model, your browser keeps it. `webllm.hasModelInCache(name)` checks whether a model is saved. We ask about every model and store the answer in a `saved` column.

In [ ]:
saved = []
for name in table["name"]:
    saved.append(await webllm.hasModelInCache(name))

table["saved"] = saved
print(f"{sum(saved)} of {len(table)} models are saved in your browser.")

### 2e: Filter the table

Now narrow the table down. Change the settings at the top of the cell and run it again:

- **`MODEL_TYPE`**: which kind of model to show (`"chat"`, `"embedding"` or `"vision"`).
- **`MAX_SIZE_MB`**: only show models that need at most this much graphics memory. **Start with 2000** (about 2 GB). Bigger models give better answers but download slowly and can be painfully slow to run on a laptop. If 2 GB models run well on your computer, try 4000.
- **`NEWER_THAN`**: only show models released in or after this month. Models with a `?` date are kept, since we do not know when they came out.

Each model also comes in a few versions. We only keep the **`q4f16`** version, which is compressed to about a quarter of the original size and is the best choice for laptops.

Each filter line keeps only the rows where the condition is true. The list is sorted newest first. To pick a model in Step 3, copy its `name`.

In [ ]:
MODEL_TYPE = "chat"
MAX_SIZE_MB = 2000
NEWER_THAN = "2025-01"

models = table
models = models[models["type"] == MODEL_TYPE]
models = models[models["name"].str.contains("q4f16")]
models = models[models["size_mb"] <= MAX_SIZE_MB]
models = models[(models["released"] >= NEWER_THAN) | (models["released"] == "?")]

# Newest first. Treat "?" as oldest so unknown dates go to the bottom.
models = models.sort_values(["released", "size_mb"], ascending=False,
                            key=lambda column: column.replace("?", ""))

print(f"{len(models)} models match your filters (newest first)")
models

### Optional: search the full model list

The table in 2e only shows models that pass your filters. To look up **any** model WebLLM supports, put a word in `SEARCH` (like `"qwen"`, `"phi"` or `"smol"`). Set it to `""` to see every model at once (it is a long list).

In [ ]:
SEARCH = "gemma"

everything = table.sort_values(["released", "size_mb"], ascending=[False, True])
if SEARCH:
    everything = everything[everything["name"].str.contains(SEARCH, case=False)]

pd.set_option("display.max_rows", None)
print(f"{len(everything)} models")
everything

## Step 3: Pick a model to download

Put the **name** of a model in `DOWNLOAD_NAME`. Copy it exactly from a table above. The default, **Gemma 3 1B** from Google, is small, quick and gives good answers, so it is a good first model.

In [ ]:
DOWNLOAD_NAME = "gemma3-1b-it-q4f16_1-MLC"

pick = table[table["name"] == DOWNLOAD_NAME]

if len(pick) == 0:
    print(f"No model is called {DOWNLOAD_NAME}. Check the spelling against the table.")
else:
    print(f"You picked: {DOWNLOAD_NAME} (about {pick['size_mb'].iloc[0]} MB)")

## Step 4: Download it

This saves the model in your browser. If it is already saved, nothing is downloaded. You can repeat Steps 3 and 4 to download more than one model.

### 4a: Set up the model loader

Downloading and loading both use the same tool, so we set it up once here.

- `engine` is the part of WebLLM that actually runs a model. It starts as `None` (Python for "nothing yet") and gets created the first time we need it.
- `loaded_model` remembers the name of the model that is currently loaded, so later steps can check that one is ready.
- `progress_bar()` shows a bar that fills up while a model downloads or loads. WebLLM (in JavaScript) calls our Python function `show` with progress updates, so `pyjs.create_callable` gives JavaScript its own handle on that function.
- `start_model()` gets a model ready to run. If the model is not saved in your browser yet, it downloads it first.

You do not need to follow every line here. Just run it.

In [ ]:
engine = None        # the thing that runs a model
loaded_model = None  # which model is currently loaded


def progress_bar():
    """Show a progress bar that updates in place."""
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    # JavaScript needs its own handle on the Python function to call it
    js_show, handle = pyjs.create_callable(show)
    return status, js_show, handle


async def start_model(model_id):
    """Get a model ready to run. Downloads it only if it is not already saved."""
    global engine
    status, progress, handle = progress_bar()

    if engine is None:
        engine = webllm.MLCEngine.new()   # create an empty engine
    engine.setInitProgressCallback(progress)

    try:
        await engine.reload(model_id)
    except Exception as error:
        # Some models have two conflicting memory-window settings.
        # If so, turn off the "sliding window" one and try again.
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            settings = js.JSON.parse(json.dumps({"sliding_window_size": -1}))
            await engine.reload(model_id, settings)
        else:
            raise

print("Model loader is ready.")

### 4b: Download your model

The `if` checks whether the model is already saved in your browser. If it is, there is nothing to do. If not (`else`), `start_model()` downloads it.

`start_model()` also loads the model onto your graphics chip, which takes up memory. Since we only wanted to download for now, `engine.unload()` frees that memory again. You will load the model you want to chat with in Step 6.

In [ ]:
if await webllm.hasModelInCache(DOWNLOAD_NAME):
    print(f"{DOWNLOAD_NAME} is already saved. Nothing to download.")
else:
    await start_model(DOWNLOAD_NAME)
    await engine.unload()   # downloaded and saved; free up memory for now
    loaded_model = None
    print(f"{DOWNLOAD_NAME} is downloaded and saved in your browser.")

## Step 5: Check what you have saved

This checks again which models are saved in your browser (you may have just downloaded one), updates the `saved` column, and shows only the saved rows.

In [ ]:
saved = []
for name in table["name"]:
    saved.append(await webllm.hasModelInCache(name))
table["saved"] = saved

saved_models = table[table["saved"]]
print(f"{len(saved_models)} models saved, about {saved_models['size_mb'].sum()} MB in total")
saved_models

**Optional: free up space.** Put the name of a saved model (from the list just above) in `DELETE_NAME` and run the cell to remove it. Leave it as `""` to do nothing.

In [ ]:
DELETE_NAME = ""

if DELETE_NAME == "":
    print("Nothing deleted.")
elif not await webllm.hasModelInCache(DELETE_NAME):
    print(f"{DELETE_NAME} is not saved, so there is nothing to delete.")
else:
    await webllm.deleteModelAllInfoInCache(DELETE_NAME)
    print(f"Deleted {DELETE_NAME}. Run Step 5 again to see the updated list.")

## Step 6: Load a model

Put the **name** of a saved model from Step 5 in `LOAD_NAME`. Loading takes a few seconds while the model moves onto your graphics chip. No downloading happens here.

To switch models later, change `LOAD_NAME` and run this cell again.

In [ ]:
LOAD_NAME = "gemma3-1b-it-q4f16_1-MLC"

if not await webllm.hasModelInCache(LOAD_NAME):
    print(f"{LOAD_NAME} is not saved yet. Download it in Steps 3 and 4 first.")
else:
    await start_model(LOAD_NAME)
    loaded_model = LOAD_NAME
    print(f"{loaded_model} is loaded and ready!")

## Step 7: Talk to the model, one piece at a time

To get an answer, we send the model a **request**: a package of data with our messages plus some settings. The request follows a fixed format (an **API**, explained in 7a), so the model knows where to find each part.

Everything we send, and everything the model writes back, is counted in **tokens**. A token is a small chunk of text, often a short word or part of a longer word. Models read and write tokens, not whole words, and their limits (how much they can read, how much they write) are measured in tokens.

We will go one piece at a time: **write the messages**, **choose the settings**, **translate the request for JavaScript**, **send it**, and **read the reply**. Then we bundle it all into a shortcut.

### Before you start: check that a model is loaded

Step 7 needs a model loaded in Step 6. This cell checks.

In [ ]:
if loaded_model is None:
    print("No model is loaded yet. Go back to Step 6 and run it until it says the model is ready.")
else:
    print(f"Ready to chat with {loaded_model}.")

### 7a: Write a message

A chat model reads a **list of messages**. Each message is a Python dictionary with two parts:

- `"role"`: who is speaking. `"user"` is you.
- `"content"`: what they said.

**Where does this format come from?** It is the format of OpenAI's **Chat Completions API**, the way programs send messages to ChatGPT's models. An **API** (application programming interface) is a set of rules for how one program talks to another. So many tools copied this format that it became a standard. WebLLM uses it too, so what you learn here also works with many other model tools.

**Other roles.** Besides `"user"`, there are two more roles:

- `"system"`: instructions for the model that apply to the whole conversation, like its personality or the rules it should follow. It goes first in the list, for example:

  ```python
  messages = [
      {"role": "system", "content": "You are a friendly tutor. Answer in two sentences."},
      {"role": "user", "content": "What is a black hole?"},
  ]
  ```

- `"assistant"`: the model's own earlier replies. Sending these back lets the model remember what it already said.

Small models do not always follow system messages closely, but it is worth trying.

Right now our conversation is just one message from you.

In [ ]:
messages = [
    {"role": "user", "content": "Write a poem about bike riding in Amsterdam."}
]

messages

### 7b: Choose the settings

The request is a Python dictionary. `"messages"` holds our conversation from 7a, and the other keys are settings (also called **arguments**) that control how the model writes:

- **`temperature`**: how random the answer is. `0` gives nearly the same answer every time; higher values give more variety (more on this in Step 8).
- **`max_tokens`**: the most tokens the model may write. This stops very long answers. If the answer gets cut off, `finish_reason` in 7e will say `"length"`.
- **`frequency_penalty`**: discourages repeating the same words. Small models sometimes get stuck repeating one line over and over; a value around `0.5` helps.

Look at the request below. It looks almost exactly like **JSON**, a plain-text format for data that most programming languages can read.

In [ ]:
TEMPERATURE = 0.7
MAX_TOKENS = 300
FREQUENCY_PENALTY = 0.5

request = {
    "messages": messages,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "frequency_penalty": FREQUENCY_PENALTY,
}

request

### 7c: Translate the request for JavaScript

Here is the part that looks strange at first. Our request is a **Python** dictionary, but the model runs in the browser's **JavaScript**, and JavaScript cannot read a Python dictionary directly. Both languages understand JSON, though, so `js_obj()` translates in two moves:

1. `json.dumps(d)` turns the Python dictionary into JSON text.
2. `js.JSON.parse(...)` asks JavaScript to read that text and build its own object.

So the request travels like this: **Python dictionary, then JSON text, then JavaScript object**, which we hand to WebLLM in 7d.

In [ ]:
def js_obj(d):
    """Turn a Python dict into a JavaScript object (by way of JSON text)."""
    return js.JSON.parse(json.dumps(d))


js_request = js_obj(request)
print("Request is translated and ready to send.")

### 7d: Send it and wait for the reply

**This is the most important line in the notebook.** It is the moment we actually talk to the model. Everything before it got the request ready; everything after it reads the answer.

`engine.chat.completions.create(...)` sends the request to the model. The name copies OpenAI's API, which uses the same name for the same job.

`await` means "wait here until the model finishes." The model writes its answer one token at a time, so this can take a few seconds.

In [ ]:
reply = await engine.chat.completions.create(js_request)

print("Got a reply!")

### 7e: Look inside the reply

The reply contains more than just the answer. Let's see the whole thing first.

In [ ]:
# Turn the JavaScript reply into a Python dict by way of JSON text
pprint(json.loads(js.JSON.stringify(reply)))

That is a lot! The parts worth knowing:

- **`choices[0].message.content`**: the model's actual answer.
- **`choices[0].finish_reason`**: why it stopped. `"stop"` means it finished on its own; `"length"` means it hit `max_tokens`.
- **`usage`**: how many tokens went in (`prompt_tokens`) and came out (`completion_tokens`).

Now we dig the answer out of that messy reply. `reply.choices[0].message.content` follows a path through it, one step at a time:

- `reply.choices` is the list of answers (there is usually just one).
- `[0]` takes the first answer.
- `.message` is that answer's message.
- `.content` is the text the model wrote.

Be patient: each step reaches across from Python into JavaScript, so this cell can take a few seconds to print.

In [ ]:
answer = reply.choices[0].message.content
print(answer)

Now let's dig the other useful parts out of the same reply: why the model stopped, how many tokens went in and came out, and how fast the model wrote. Each line follows a path through the reply, just like the answer did.

**Tokens per second** is the model's writing speed. WebLLM times itself and keeps the result under `usage.extra`. Notebook 02 shows how to measure it yourself and compare models.

In [ ]:
print("Why it stopped:   ", reply.choices[0].finish_reason)
print("Tokens you sent:  ", reply.usage.prompt_tokens)
print("Tokens it wrote:  ", reply.usage.completion_tokens)
print("Tokens per second:", round(reply.usage.extra.decode_tokens_per_s, 1))

### 7f: Bundle it into a shortcut

Doing all of that every time would be tedious. Here we put steps 7a to 7e into one function called `ask()`. Give it a question, and it gives back the answer. It uses the same settings as 7b, and you can change `temperature` or `max_tokens` each time you call it.

In [ ]:
async def ask(question, temperature=0.7, max_tokens=MAX_TOKENS):
    if loaded_model is None:
        return "No model is loaded yet. Run Step 6 until it says the model is ready."

    # 7a: write the messages
    messages = [{"role": "user", "content": question}]

    # 7b: choose the settings
    request = {
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "frequency_penalty": FREQUENCY_PENALTY,
    }

    # 7c and 7d: translate the request and send it to the model
    reply = await engine.chat.completions.create(js_obj(request))

    # 7e: read the answer
    return reply.choices[0].message.content


print(await ask("Write a poem about bike riding in Amsterdam."))

## Step 8: Experiment with temperature

Temperature controls randomness. Low temperature gives similar answers every time; high temperature gives more varied (and sometimes stranger) answers. Run this cell a few times.

`SHORT_ANSWER` sets `max_tokens` to 50 here to keep the answers short, so we can compare them quickly. If an answer stops mid-sentence, it ran out of tokens.

In [ ]:
SHORT_ANSWER = 50

question = "Write a one-line slogan for a coffee shop."

for t in [0.0, 0.7, 1.5]:
    answer = await ask(question, temperature=t, max_tokens=SHORT_ANSWER)
    print(f"temperature={t}: {answer}\n")

## Step 9: Watch the model write (streaming)

So far we waited for the whole answer and then printed it. But the model writes one token at a time, so we can also show each piece as soon as it is written. This is called **streaming**, and it is how chat websites make the words appear as you watch.

To turn it on, add `"stream": True` to the request. This is part of the same API format as before.

### 9a: Send a streaming request and read the first piece

With streaming on, `create()` does not give back a finished reply. It gives back a **stream**: a source of small pieces (called **chunks**) that arrive one after another.

We read the stream with a **reader** that hands out one chunk each time we call `reader.next()`. Python cannot loop over a JavaScript stream directly, so the `js.Function(...)` line asks JavaScript to make the reader for us.

Let's read just the first chunk and look inside it. Like the reply in 7e, we turn it into a Python dictionary by way of JSON text.

In [ ]:
STREAM_QUESTION = "Describe a sunrise over the ocean in a short paragraph."

request = {
    "messages": [{"role": "user", "content": STREAM_QUESTION}],
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "frequency_penalty": FREQUENCY_PENALTY,
    "stream": True,
}

stream = await engine.chat.completions.create(js_obj(request))
reader = js.Function("stream", "return stream[Symbol.asyncIterator]()")(stream)

step = await reader.next()
first_chunk = json.loads(js.JSON.stringify(step.value))
pprint(first_chunk)

Each chunk looks a lot like the reply from 7e, but instead of `message` it has **`delta`**, which means "the change": only the new bit of text in this chunk, in `choices[0].delta.content`.

### 9b: Read the rest of the stream

Now keep calling `reader.next()` until the stream says it is `done`. `chunk_text()` pulls out the new bit of text, and gives `""` for a chunk with no text (the last chunk is often empty). Each time, we print the new bit of text right away. `end=""` keeps everything on one line instead of starting a new line each time, and `flush=True` shows it immediately.

In [ ]:
def chunk_text(chunk):
    """The new bit of text in one chunk ("" if there is none)."""
    if not chunk["choices"]:
        return ""
    return chunk["choices"][0]["delta"].get("content") or ""


print(chunk_text(first_chunk), end="", flush=True)

while True:
    step = await reader.next()
    if step.done:
        break
    chunk = json.loads(js.JSON.stringify(step.value))
    print(chunk_text(chunk), end="", flush=True)

print("\n\n(Stream finished.)")

## Your turn

Try your own questions below. Where is this model good? Where does it make mistakes? Load a different model in Step 6, ask the same question, and compare.

In [ ]:
print(await ask("What is the capital of Australia?"))